# Train on Colab GPU

Launcher notebook — clones the repo, pulls data from Drive, and runs training.
All code lives in the repo, this notebook just invokes it.

**Before running:**
1. Set your GitHub token in Cell 1
2. Select a GPU runtime (Runtime > Change runtime type > T4/A100)
3. Ensure HDF5 data files exist in `MyDrive/masters-thesis/data/`

**Drive layout:**
```
MyDrive/
  masters-thesis/
    data/                          <- train.h5, val.h5, test.h5 (uploaded once)
    runs/
      egnn/20260411_143022/        <- checkpoints/ and logs/ per run
      sweep/20260416/              <- all sweep run checkpoints and logs
```

**Modes:**
- Cell 4a: single training run (one config)
- Cell 4b: hyperparameter sweep (grid search over lr and noise_factor)

In [ ]:
# Cell 1: Mount Drive + clone repo
from google.colab import drive
drive.mount("/content/drive")

GITHUB_TOKEN = ""  # paste your token here (never commit this)
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
DRIVE = "/content/drive/MyDrive/masters-thesis"

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git repo
%cd repo

In [ ]:
# Cell 2: Install dependencies (torch is pre-installed on Colab)
!pip install -q -r requirements.txt

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Cell 3: Pull data from Drive
!mkdir -p data/output
!cp {DRIVE}/data/train.h5 data/output/
!cp {DRIVE}/data/val.h5 data/output/
!cp {DRIVE}/data/test.h5 data/output/

import h5py
with h5py.File("data/output/train.h5", "r") as f:
    print(f"train: {f['trajectories'].shape}")

In [ ]:
# Cell 4a: Single training run
from datetime import datetime

MODEL = "egnn"  # change to "hgnn" for HGNN
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run: {MODEL}/{RUN_ID}")

!python -m training.train --config configs/{MODEL}.yaml

In [ ]:
# Cell 4b: Hyperparameter sweep (run this INSTEAD of Cell 4a)
# Grid: lr x noise_factor, aggressive configs first
# Default 200 epochs per run, 9 runs total

!python -m training.sweep --epochs 200

In [ ]:
# Cell 5: Save results to Drive
import os
from datetime import datetime

MODEL = "egnn"
SAVE_TAG = datetime.now().strftime("%Y%m%d")
RUN_DIR = f"{DRIVE}/runs/{MODEL}/{SAVE_TAG}"

!mkdir -p {RUN_DIR}
!cp -r checkpoints/{MODEL}/ {RUN_DIR}/checkpoints/ 2>/dev/null || true
!cp -r logs/{MODEL}/ {RUN_DIR}/logs/ 2>/dev/null || true

n_runs = len(os.listdir(f"checkpoints/{MODEL}/")) if os.path.exists(f"checkpoints/{MODEL}/") else 0
print(f"Saved {n_runs} run(s) to {RUN_DIR}")